### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 (심화) — 멀티 에이전트 4대 실패 모드와 방어
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar)**

> 목표: 협업 구조를 '구현'하는 데서 그치지 않고, 강의가 경고한 **① 무한 루프 · ② 교착(Deadlock) · ③ 비용 폭증 · ④ 행동 붕괴**를 직접 재현하고 가드로 막습니다.
> 이전 강의에서 5가지 협업 토폴로지를 만들었다면, 이번에는 그 구조가 **어떻게 실패하는지**와 **무엇으로 막는지**를 다룹니다.

## 0. 셋업
실패 모드 대부분은 **결정론적 재현**을 위해 가짜(mock) 에이전트로 시연하고, '행동 붕괴'만 실제 Solar 호출로 확인합니다. Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`를 등록한 뒤 실행합니다.

In [1]:
!pip -q install openai

In [2]:
# Upstage Solar 셋업 — Colab 좌측 🔑 Secrets에 'UPSTAGE_API_KEY'를 등록한 뒤 실행합니다.
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
from openai import OpenAI
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro2"

def ask(system, user, **kw):
    """Solar 단일 호출 헬퍼."""
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}], **kw)
    return r.choices[0].message.content.strip()
print("Solar 준비 완료:", MODEL)

Solar 준비 완료: solar-pro2


## 1. 실패 모드 ① 무한 루프 (Loop)
두 에이전트가 서로에게 끝없이 일을 떠넘기면 시스템이 멈추지 않습니다. **스텝 예산(max_steps)**으로 상한을 둬 차단합니다.

In [3]:
class StepBudget:
    def __init__(self, max_steps):
        self.max_steps = max_steps; self.steps = 0
    def charge(self):
        self.steps += 1
        if self.steps > self.max_steps:
            raise RuntimeError(f"🛑 max_steps({self.max_steps}) 초과 — 무한 루프 차단 (현재 {self.steps}스텝)")

def ping_pong(budget):
    # A는 항상 B에게, B는 항상 A에게 다시 위임 → 끝나지 않는 구조
    turn = "A"
    while True:
        budget.charge()
        nxt = "B" if turn == "A" else "A"
        print(f"  {turn} → {nxt} 에게 위임")
        turn = nxt

# 가드가 없으면 영원히 반복됩니다 → 스텝 예산으로 강제 종료
try:
    ping_pong(StepBudget(max_steps=5))
except RuntimeError as e:
    print("가드 작동:", e)

  A → B 에게 위임
  B → A 에게 위임
  A → B 에게 위임
  B → A 에게 위임
  A → B 에게 위임
가드 작동: 🛑 max_steps(5) 초과 — 무한 루프 차단 (현재 6스텝)


## 2. 실패 모드 ② 교착 (Deadlock)
A는 B의 결과를, B는 A의 결과를 기다리면 **둘 다 영원히 대기**합니다(순환 대기). 작업 의존 그래프에서 **사이클을 탐지**해 미리 막습니다.

In [4]:
def detect_cycle(deps):
    # deps: {task: [선행 task들]} — 사이클이 있으면 교착
    WHITE, GRAY, BLACK = 0, 1, 2
    color = {t: WHITE for t in deps}
    found = []
    def dfs(t, path):
        color[t] = GRAY
        for nxt in deps.get(t, []):
            if color.get(nxt, WHITE) == GRAY:        # 회색 노드 재방문 = 사이클
                found.extend(path + [t, nxt]); return True
            if color.get(nxt, WHITE) == WHITE and dfs(nxt, path + [t]):
                return True
        color[t] = BLACK
        return False
    for t in deps:
        if color[t] == WHITE and dfs(t, []):
            return found
    return None

# A는 B를, B는 C를, C는 다시 A를 기다림 → 순환 대기(교착)
deadlock = {"A": ["B"], "B": ["C"], "C": ["A"]}
cyc = detect_cycle(deadlock)
print("교착 탐지:", " → ".join(cyc) if cyc else "없음")

# 의존을 끊으면(C가 A를 기다리지 않게) 안전해집니다
safe = {"A": ["B"], "B": ["C"], "C": []}
print("수정 후:", "교착 없음 ✅" if detect_cycle(safe) is None else "여전히 교착")

교착 탐지: A → B → C → A
수정 후: 교착 없음 ✅


## 3. 실패 모드 ③ 비용 폭증 (Cost Explosion)
멀티 에이전트는 LLM 호출 수가 곱셈으로 늘어 비용이 폭증합니다. 토폴로지별 **호출 수·토큰을 정량 비교**해 어디가 비싼지 봅니다.

In [5]:
def estimate(topology, n_topics=3, rounds=3):
    # (호출 수, 토큰) 추정 — 호출당 평균 400토큰으로 가정
    if topology == "sequential":
        calls = 3 * n_topics                 # 토픽마다 research→analyze→write
    elif topology == "parallel":
        calls = 3 * n_topics + 1             # + 최종 종합 1회
    elif topology == "feedback":
        calls = 3 + rounds * 2               # 초안 + (critic+rewrite)*rounds
    elif topology == "naive_recursive":
        calls = 2 ** (rounds + 1)            # 가드 없는 재귀 → 지수 폭증
    else:
        calls = 0
    return calls, calls * 400

for topo in ["sequential", "parallel", "feedback", "naive_recursive"]:
    calls, tok = estimate(topo)
    print(f"  {topo:16s} 호출 {calls:3d}  토큰 {tok:6d}  {'█' * (calls // 2)}")
print("\n→ 가드 없는 재귀(naive_recursive)는 지수적으로 폭증합니다. max_steps/max_depth가 필수입니다.")

  sequential       호출   9  토큰   3600  ████
  parallel         호출  10  토큰   4000  █████
  feedback         호출   9  토큰   3600  ████
  naive_recursive  호출  16  토큰   6400  ████████

→ 가드 없는 재귀(naive_recursive)는 지수적으로 폭증합니다. max_steps/max_depth가 필수입니다.


## 4. 실패 모드 ④ 행동 붕괴 (Behavioral Collapse)
같은 전략만 반복하면 에이전트의 출력 **다양성이 무너집니다**. 동일 프롬프트를 반복 생성해 다양성을 측정하고, **페르소나를 강제 회전**해 회복시킵니다. (실제 Solar 호출)

In [6]:
import itertools

def diversity(texts):
    # 출력 쌍의 평균 자카드 유사도가 높을수록 다양성이 낮음 → 1 - 평균유사도
    pairs = list(itertools.combinations(texts, 2))
    if not pairs:
        return 1.0
    sims = []
    for a, b in pairs:
        A, B = set(a.split()), set(b.split())
        sims.append(len(A & B) / len(A | B) if (A | B) else 0)
    return round(1 - sum(sims) / len(sims), 2)

topic = "AI 에이전트의 장점을 한 문장으로 말해줘"

# (붕괴) 동일 페르소나로 5번 → 비슷한 답만 반복
same = [ask("너는 IT 칼럼니스트입니다.", topic, temperature=0.2) for _ in range(5)]
print("다양성(동일 페르소나):", diversity(same))

# (회복) 페르소나를 강제 회전 → 관점이 갈라져 다양성 회복
personas = ["보수적인 CFO", "낙관적인 스타트업 창업가", "회의적인 보안 전문가",
            "현장 운영자", "투자 분석가"]
varied = [ask(f"너는 {p}입니다.", topic, temperature=0.2) for p in personas]
print("다양성(페르소나 회전):", diversity(varied))
for p, v in zip(personas, varied):
    print(f"  - {p}: {v[:40]}")

다양성(동일 페르소나): 0.46
다양성(페르소나 회전): 0.9
  - 보수적인 CFO: AI 에이전트는 데이터 기반 의사결정, 업무 자동화, 운영 효율성 향상,
  - 낙관적인 스타트업 창업가: "AI 에이전트는 24/7 운영, 데이터 기반 의사결정, 반복 작업 자동
  - 회의적인 보안 전문가: "AI 에이전트는 반복적이고 규칙 기반의 작업을 효율적으로 처리할 수 있
  - 현장 운영자: "AI 에이전트는 24/7 운영, 대규모 데이터 실시간 분석, 인간보다 
  - 투자 분석가: "AI 에이전트는 데이터 처리, 패턴 분석, 예측, 자동화 등을 통해 인


## 직접 해보기
위 비용 추정에 **라우터(router)** 토폴로지를 추가해 보세요. 라우터는 요청 1건당 '라우팅 결정 1회 + 실행 1회'로 가정합니다.

In [7]:
# TODO: estimate() 에 topology == 'router' 분기를 추가하세요.
#       router 는 요청 1건당 calls = 2 (라우팅 1 + 실행 1) 로 가정합니다.

In [8]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# def estimate_router(n_requests=3):
#     calls = 2 * n_requests
#     return calls, calls * 400
# print('router', estimate_router())

## 정리 — 4대 실패 모드와 방어
| 실패 모드 | 증상 | 방어 |
|---|---|---|
| ① 무한 루프 | 종료되지 않는 위임 | `max_steps` 스텝 예산 |
| ② 교착(Deadlock) | 순환 대기로 전원 정지 | 의존 그래프 **사이클 탐지** |
| ③ 비용 폭증 | 호출 수 곱셈/지수 증가 | `max_depth`·예산·정량 모니터링 |
| ④ 행동 붕괴 | 동일 전략만 반복 | **다양성 강제**(페르소나 회전) |

- 협업 구조는 강력하지만 **가드가 전제**입니다 — 한도와 탐지를 먼저 설계합니다.
- 비용·재귀 폭주는 이후 강의의 회로 차단기·타임아웃으로 더 단단히 막습니다.